(course-enzyme-energy-minimization)=

# Path B - Module 43: Energy Minimization

Experimental structures and engineered mutants often have high-energy spots: atoms that clash or bonds with unnatural lengths. Such defects can make integration unstable; preparation must establish a consistent physical model first. 

In this module, you will learn to use MolSysMT to **Relax** your engineered PETase and reduce the energy of a prepared model by local optimization.

The PETase/BHET recipe requires compatible protein and ligand parameters, hydrogen/chemical-state assignments and a deliberate substrate placement. A SMILES conversion plus merge does not establish a binding pose. Select one structure explicitly before energy evaluation.

For a native `MolSys`, the default call returns an independent system. The examples retain it in `molsys`, so later analysis uses the minimized coordinates. To edit a native input deliberately, use `in_place=True` and retain the existing variable; that mode returns `None`. The supported method is L-BFGS with engine default stopping settings; this public API has no `tolerance` argument. See {ref}`the minimization tutorial <Tutorial_Potential_energy_minimization>`.

An existing OpenMM Context/Simulation cannot currently be copied independently. Such requests, including explicit output forms, are rejected before optimization. Use the native independent workflow above, or explicitly choose in-place engine editing when that ownership is intended.


Before constructing a mechanical model, inspect the [effective model recipe](../../user/tools/molecular_mechanics/get_model_recipe.md). Stored unsupported tuning requests now refuse with named-field diagnostics; they are not silently applied. The recipe distinguishes physical controls from execution settings and does not prepare chemistry or change coordinates.


In [ ]:
import molsysmt as msm
from molsysmt import systems

# Load our complex (Enzyme + BHET)
petase = msm.convert('pdb:6EQE', to_form='molsysmt.MolSys', selection='molecule_type=="protein"')
msm.build.add_missing_hydrogens(petase, pH=8.0)
bhet = msm.convert('C1=CC(=CC=C1C(=O)OCCO)C(=O)OCCO', from_form='string:smiles', to_form='molsysmt.MolSys')
molsys = msm.merge([petase, bhet])
molsys = msm.extract(molsys, structure_indices=0)

# Initial energy (likely high)
e_start = msm.molecular_mechanics.get_potential_energy(molsys)
print(f"Energy before relaxation: {e_start}")

## 1. The Relaxation Engine
The implemented method is L-BFGS local optimization, with the engine's default stopping and constraint settings. The calculation searches from the initial structure; it does not promise the nearest or global minimum, and constrained atom forces need not all vanish.


In [ ]:
# Run energy minimization
molsys_before = molsys
molsys = msm.molecular_mechanics.potential_energy_minimization(molsys)

print("Minimization finished.")

## 2. Monitoring the Drop
Compare finite energies under the same force field, constraints and unit convention. The size and sign of a change depend on the prepared model; there is no universal expected drop. Local optimization and a lower energy do not establish binding free energy, equilibrium or physical stability.


In [ ]:
e_final = msm.molecular_mechanics.get_potential_energy(molsys)
print(f"Energy after relaxation: {e_final}")
print(f"Difference: {e_start - e_final}")

---

## 🏆 Path B Challenge: The Stress Buster

1. Inspect the force vectors before and after minimization with consistent units.
2. Interpret residual forces together with constraints and the engine stopping criterion; a maximum atom force is not the optimizer's RMS force criterion.
3. Retain both systems, align them and measure RMSD without assuming a universal small-change threshold.
4. Inspect secondary-structure changes rather than assuming that minimization must preserve them.

The resulting configuration has been optimized locally under the chosen model. Continue with [Module 44: AMBER TLeap](44_AMBER_TLeap.ipynb) to inspect parameter assignment; minimization already requires a valid force-field model.


(course-enzyme-energy-minimization-see-also)=
:::{seealso}
:class: dropdown

- {ref}`Minimization tutorial <Tutorial_Potential_energy_minimization>` — result ownership, units and supported options.
- {ref}`Potential energy <Tutorial_Get_potential_energy>` — compare a prepared model before and after optimization.
- [Module 44: AMBER TLeap](44_AMBER_TLeap.ipynb) — force-field assignment.
:::
